In [ ]:
!pip -q install -U transformers peft datasets accelerate huggingface_hub
!pip install -U "torchao>=0.16.0"
!pip -q install peft sacrebleu langdetect immutabledict "nltk>=3.9.1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
import transformers
import peft
import datasets
import huggingface_hub

print("PyTorch:", torch.__version__)
print("GPU :", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

PyTorch: 2.11.0+cu130
GPU : True
Transformers: 5.20.2
PEFT: 0.21.2
Datasets: 5.1.0
Hugging Face Hub: 2.3.0


In [ ]:
from huggingface_hub import login
from google.colab import drive

login()
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile plain_train.py

import os
import json
import time
import csv
import argparse
import random
import contextlib
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType

# ==========================================
# 0. CLI ARGUMENTS & CONFIG
# ==========================================
# Same template as the LLM-JEPA script. The ONLY difference is the training
# objective: plain LM loss (no JEPA passes, no JEPA loss).
parser = argparse.ArgumentParser(description="Plain LM fine-tuning (LoRA) - same config as the LLM-JEPA script, JEPA removed")
parser.add_argument("--seed", type=int, default=42, help="Random seed")
parser.add_argument("--arm_name", type=str, default=None, help="Custom output directory name")

parser.add_argument("--bs", type=int, default=4, help="Micro-batch size pushed through the GPU (default: 4)")
parser.add_argument("--accum", type=int, default=4, help="Gradient accumulation steps; effective batch = bs * accum (default: 16)")
parser.add_argument("--max_len", type=int, default=256, help="Token limit for full LM seq, context view and target view; pair dropped if any exceeds it")
parser.add_argument("--n_total", type=int, default=50_000, help="Number of raw pairs read from the dataset")
parser.add_argument("--lr", type=float, default=2e-4, help="Learning rate")
parser.add_argument("--model", type=str, default="meta-llama/Llama-3.2-1B-Instruct", help="HF model id or local path")
parser.add_argument("--dataset_dir", type=str, default="/content/drive/MyDrive/SIFT_dataset", help="Folder with opus100.json / opus100.csv")
parser.add_argument("--out_root", type=str, default="/content/drive/MyDrive/adapters", help="Each arm is saved to <out_root>/<arm_name>")
parser.add_argument("--precision", type=str, default="fp16", choices=["fp16", "bf16"], help="Mixed-precision mode. MUST be identical for the plain and JEPA arms.")
args = parser.parse_args()

if args.bs < 1 or args.accum < 1:
    raise ValueError("--bs and --accum must be positive.")

ARM_NAME = args.arm_name or "plain_fine_tuning"
OUT = os.path.join(args.out_root, ARM_NAME)
os.makedirs(OUT, exist_ok=True)

MODEL_DIR = args.model

OPUS100_CSV_PATH = os.path.join(args.dataset_dir, "opus100.csv")
OPUS100_JSON_PATH = os.path.join(args.dataset_dir, "opus100.json")

# Hyperparameters
N_TOTAL = args.n_total
MAX_LEN = args.max_len
BS = args.bs
ACCUM = args.accum
LR = args.lr

# Precision (fp16 uses a GradScaler; bf16 does not). Keep identical across both arms.
USE_FP16 = args.precision == "fp16"
AMP_DTYPE = torch.float16 if USE_FP16 else torch.bfloat16

# NOT a training setting. This only makes the length filter count the [PRED]
# token in the context view (the JEPA k=1 default) so the plain arm keeps exactly
# the same pairs as the JEPA arm. Set to 0 only if your JEPA runs use k=0.
LEN_FILTER_K = 1

random.seed(args.seed)
torch.manual_seed(args.seed)

# ==========================================
# 1. LOCAL DATA LOADERS & TOKENIZATION
# ==========================================
def load_local_opus100(n_samples):
    pairs = []
    if os.path.exists(OPUS100_JSON_PATH):
        print(f"Loading OPUS-100 from: {OPUS100_JSON_PATH}")
        with open(OPUS100_JSON_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
            data_slice = data[:n_samples] if isinstance(data, list) else data.get("train", [])[:n_samples]
            for item in data_slice:
                en = item.get("en") or item.get("english") or item.get("translation", {}).get("en")
                vi = item.get("vi") or item.get("vietnamese") or item.get("translation", {}).get("vi")
                if en and vi:
                    pairs.append((str(en), str(vi)))
    elif os.path.exists(OPUS100_CSV_PATH):
        print(f"Loading OPUS-100 from: {OPUS100_CSV_PATH}")
        df = pd.read_csv(OPUS100_CSV_PATH).head(n_samples)
        for _, row in df.iterrows():
            en = row.get("en") or row.get("english")
            vi = row.get("vi") or row.get("vietnamese")
            if pd.notna(en) and pd.notna(vi):
                pairs.append((str(en), str(vi)))
    else:
        raise FileNotFoundError(
            f"Could not find {OPUS100_JSON_PATH} or {OPUS100_CSV_PATH}. "
            "In Colab, mount Drive first: from google.colab import drive; drive.mount('/content/drive')"
        )
    return pairs

def chat(tok, text):
    return tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)

def encode_jepa_pair(tok, user_lm, ans_lm, en, vi, pred_id, k, need_jepa):
    # 1. LM Pass (Direction Alternating)
    p_ids = tok(chat(tok, user_lm), add_special_tokens=False).input_ids
    full_ids = tok(
        tok.apply_chat_template(
            [{"role": "user", "content": user_lm}, {"role": "assistant", "content": ans_lm}],
            tokenize=False
        ),
        add_special_tokens=False
    ).input_ids

    full_labels = [-100] * len(p_ids) + full_ids[len(p_ids):]

    ctx_ids, tgt_ids = [], []
    if need_jepa:
        # 2. JEPA Views (FIXED DIRECTION: NEW/VI is Context, EN is Target)
        ctx_ids = tok(chat(tok, vi), add_special_tokens=False).input_ids
        if k == 1:
            ctx_ids.append(pred_id)  # Append [PRED] token for k=1

        tgt_ids = tok(chat(tok, en), add_special_tokens=False).input_ids  # Symmetric wrapper

    if len(full_ids) > MAX_LEN or (need_jepa and (len(ctx_ids) > MAX_LEN or len(tgt_ids) > MAX_LEN)):
        return None

    return {
        "full_ids": full_ids,
        "full_labels": full_labels,
        "ctx_ids": ctx_ids,
        "tgt_ids": tgt_ids
    }

def build_data(tok, pred_id, k, need_jepa):
    raw_pairs = load_local_opus100(N_TOTAL)
    items = []

    for i, (en, vi) in enumerate(raw_pairs):
        if i % 2 == 0:
            u_msg = f"Translate the following text to Vietnamese. Output only the translation.\n\n{en}"
            a_msg = vi
        else:
            u_msg = f"Translate the following text to English. Output only the translation.\n\n{vi}"
            a_msg = en
        items.append((u_msg, a_msg, en, vi))

    random.shuffle(items)

    processed = []
    for u, a, en, vi in items:
        enc = encode_jepa_pair(tok, u, a, en, vi, pred_id, k, need_jepa)
        if enc:
            processed.append(enc)

    print(f"Total processed samples (k={k}, JEPA active={need_jepa}): {len(processed)}")
    return processed

def collate_jepa(rows, pad_token_id, need_jepa):
    # Collate Full LM
    max_full = max(len(r["full_ids"]) for r in rows)
    full_ids = torch.full((len(rows), max_full), pad_token_id)
    full_lab = torch.full((len(rows), max_full), -100)
    full_mask = torch.zeros((len(rows), max_full), dtype=torch.long)

    for i, r in enumerate(rows):
        full_ids[i, :len(r["full_ids"])] = torch.tensor(r["full_ids"])
        full_lab[i, :len(r["full_labels"])] = torch.tensor(r["full_labels"])
        full_mask[i, :len(r["full_ids"])] = 1

    batch = {
        "full_ids": full_ids.cuda(),
        "full_mask": full_mask.cuda(),
        "full_lab": full_lab.cuda()
    }

    if need_jepa:
        # Collate Context (VI + optional [PRED])
        max_ctx = max(len(r["ctx_ids"]) for r in rows)
        ctx_ids = torch.full((len(rows), max_ctx), pad_token_id)
        ctx_mask = torch.zeros((len(rows), max_ctx), dtype=torch.long)
        ctx_last_idx = []

        # Collate Target (EN)
        max_tgt = max(len(r["tgt_ids"]) for r in rows)
        tgt_ids = torch.full((len(rows), max_tgt), pad_token_id)
        tgt_mask = torch.zeros((len(rows), max_tgt), dtype=torch.long)
        tgt_last_idx = []

        for i, r in enumerate(rows):
            ctx_ids[i, :len(r["ctx_ids"])] = torch.tensor(r["ctx_ids"])
            ctx_mask[i, :len(r["ctx_ids"])] = 1
            ctx_last_idx.append(len(r["ctx_ids"]) - 1)

            tgt_ids[i, :len(r["tgt_ids"])] = torch.tensor(r["tgt_ids"])
            tgt_mask[i, :len(r["tgt_ids"])] = 1
            tgt_last_idx.append(len(r["tgt_ids"]) - 1)

        batch.update({
            "ctx_ids": ctx_ids.cuda(),
            "ctx_mask": ctx_mask.cuda(),
            "ctx_last_idx": torch.tensor(ctx_last_idx).cuda(),
            "tgt_ids": tgt_ids.cuda(),
            "tgt_mask": tgt_mask.cuda(),
            "tgt_last_idx": torch.tensor(tgt_last_idx).cuda()
        })

    return batch

# ==========================================
# 2. TRAINING ENGINE
# ==========================================
def train(model, tok, pred_id):
    # The JEPA views are built here ONLY so the length filter (full LM seq,
    # context view, target view) keeps exactly the same pairs as the JEPA arm.
    # They are never used in training: collate below is called with need_jepa=False.
    data = build_data(tok, pred_id, LEN_FILTER_K, True)
    total_steps = len(data) // (BS * ACCUM)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, int(0.03 * total_steps), total_steps)

    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)  # no-op when bf16
    ck = {max(1, int(total_steps * f)): int(f * 100) for f in (0.25, 0.50, 0.75, 1.0)}

    # Same CSV columns as the JEPA run so both logs can be analysed together.
    # JEPA / anisotropy columns are 0 here (same as a JEPA run with lbd=0).
    f_log = open(f"{OUT}/train_log.csv", "w", newline="")
    w_log = csv.writer(f_log)
    w_log.writerow(["step", "loss", "lm_loss", "jepa_loss", "ctx_std", "ctx_pair", "tgt_std", "tgt_pair", "lr", "grad_norm"])

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    idx = 0
    model.train()

    for step in range(1, total_steps + 1):
        tot_loss_accum = 0.0
        tot_lm_accum = 0.0
        lr = opt.param_groups[0]["lr"]

        for _ in range(ACCUM):
            batch = collate_jepa(data[idx:idx + BS], tok.pad_token_id, False)
            idx += BS

            with torch.autocast("cuda", dtype=AMP_DTYPE):
                # Causal LM Pass (the only loss in the plain arm)
                out_lm = model(input_ids=batch["full_ids"], attention_mask=batch["full_mask"], labels=batch["full_lab"])
                lm_loss = out_lm.loss
                total_loss = lm_loss

            scaler.scale(total_loss / ACCUM).backward()

            tot_loss_accum += total_loss.item() / ACCUM
            tot_lm_accum += lm_loss.item() / ACCUM

        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        opt.zero_grad(set_to_none=True)

        w_log.writerow([
            step,
            f"{tot_loss_accum:.6f}",
            f"{tot_lm_accum:.6f}",
            f"{0.0:.6f}",
            f"{0.0:.6f}",
            f"{0.0:.6f}",
            f"{0.0:.6f}",
            f"{0.0:.6f}",
            f"{lr:.8f}",
            f"{float(gn):.6f}"
        ])

        if step % 20 == 0:
            f_log.flush()
            print(f"Step {step}/{total_steps} | Loss: {tot_loss_accum:.4f} | LM: {tot_lm_accum:.4f}")

        if step in ck:
            pct = ck[step]
            ckpt_dir = f"{OUT}/ckpt_{pct}pct"
            print(f"\n--> Saving Checkpoint at {pct}% to {ckpt_dir}...")
            model.save_pretrained(ckpt_dir)
            tok.save_pretrained(ckpt_dir)

    f_log.close()

    train_mins = (time.time() - t0) / 60.0
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

    telemetry_data = {
        "arm_name": ARM_NAME,
        "training_type": "plain_lm",
        "len_filter_counts_pred_token": bool(LEN_FILTER_K),
        "lambda_jepa": 0.0,
        "gamma_lm": 1.0,
        "layer": None,
        "target_type": None,
        "seed": args.seed,
        "precision": args.precision,
        "micro_batch_size": BS,
        "gradient_accumulation_steps": ACCUM,
        "effective_batch_size": BS * ACCUM,
        "max_len": MAX_LEN,
        "n_train_samples": len(data),
        "total_training_steps": total_steps,
        "train_minutes": round(train_mins, 2),
        "train_peak_vram_gb": round(peak_vram, 2)
    }

    return telemetry_data

# ==========================================
# 3. MAIN EXECUTION ROUTINE
# ==========================================
if __name__ == "__main__":
    print(f"\n=======================================================")
    print(f" STARTING PLAIN LM TRAINING: {ARM_NAME.upper()}")
    print(f" Base Model Path: {MODEL_DIR}")
    print(f" Output Dir     : {OUT}")
    print(f" Parameters     : seed={args.seed} | training=plain LM loss")
    print(f" Batching       : bs={BS} x accum={ACCUM} = {BS * ACCUM} | max_len={MAX_LEN} | precision={args.precision}")
    print(f"=======================================================\n")

    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    tok.pad_token = tok.pad_token or tok.eos_token

    PRED_TOKEN = "<|reserved_special_token_0|>"
    PRED_ID = tok.convert_tokens_to_ids(PRED_TOKEN)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        torch_dtype=AMP_DTYPE,
        device_map="cuda"
    )

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    model = get_peft_model(model, peft_config)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    telemetry = train(model, tok, PRED_ID)

    telemetry_path = os.path.join(OUT, "telemetry.json")
    with open(telemetry_path, "w", encoding="utf-8") as f:
        json.dump(telemetry, f, indent=2)

    print("\n=======================================================")
    print(f"      TRAINING COMPLETED SUCCESSFULLY: {ARM_NAME}")
    print("=======================================================")
    print(f"Saved training log to    : {OUT}/train_log.csv")
    print(f"Saved 100% adapter to    : {OUT}/ckpt_100pct")
    print(f"Saved telemetry record to: {telemetry_path}")

Overwriting plain_train.py


In [ ]:
!python plain_train.py --seed 42 --bs 4 --accum 4 --precision fp16


 STARTING PLAIN LM TRAINING: PLAIN_FINE_TUNING
 Base Model Path: meta-llama/Llama-3.2-1B-Instruct
 Output Dir     : /content/drive/MyDrive/adapters/plain_fine_tuning
 Parameters     : seed=42 | training=plain LM loss
 Batching       : bs=4 x accum=4 = 16 | max_len=256 | precision=fp16

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:02<00:00, 62.98it/s]
Loading OPUS-100 from: /content/drive/MyDrive/SIFT_dataset/opus100.json
Total processed samples (k=1, JEPA active=True): 49998
Step 20/3124 | Loss: 1.9015 | LM: 1.9015
Step 40/3124 | Loss: 2.2184 | LM: 2.2184
Step 60/3124 | Loss: 1.9129 | LM: 1.9129
Step 80/3124 | Loss: 2.3927 | LM: 2.3927
Step 100/3124 | Loss: 2.1079 | LM: 2.1079
Step 120/3124 | Loss: 2.2554 | LM: 2.2554
Step 140/3124 | Loss: 2.2473 | LM: 2.2473
Step 160/3124 | Loss: 2.1286 | LM: 2.1286
Step 180/3124 | Loss: 1.5843 | LM: 1.5843
Step 200/3124 | Loss: 1.9608 | LM: 1.9608
Step 220/3124 | Loss: 2.1301 | LM: 2.1301
Step 24

In [ ]:
%%writefile eval.py
# ======================================================================================
# RUN FROM A COLAB CELL (one eval at a time):
#   !python eval.py --arm plain_fine_tuning --tests flores    # FLORES EN->VI and VI->EN (BLEU, chrF++)
#   !python eval.py --arm plain_fine_tuning --tests mlqa      # MLQA-vi (exact match, F1)
#   !python eval.py --arm plain_fine_tuning --tests ifeval    # IFEval (strict / loose)
#
# Extras:  --tests flores mlqa    several at once        --tests all   everything (default)
#          --arm base             the un-tuned model     --quick       8 examples, pipeline check
#          --force                redo finished tests    --n-flores N  random sample of N (default: all rows)
#
# Modular options (all optional; defaults reproduce the previous behaviour):
#          --ckpt NAME            which checkpoint folder inside the arm (default ckpt_100pct; falls back to the
#                                 arm folder itself if the adapter files are there)
#          --out-name NAME        results folder name inside the arm (default "eval")
#          --dtype fp16|bf16      generation precision (default fp16; use fp16 on a T4)
#          --base-model ID        base model (default meta-llama/Llama-3.2-1B-Instruct)
#          --seed N               FLORES sampling seed (default 42)
#          --bs-flores/--bs-mlqa/--bs-ifeval            generation batch sizes
#          --max-new-flores/--max-new-mlqa/--max-new-ifeval   max generated tokens
#          --flores-prompt short|train   FLORES instruction wording (default short = previous behaviour;
#                                        train = the wording the arms were trained with)
#
# ALL metrics go into ONE file:  <adapter-root>/<arm>/eval/results.json
# Each test is written into it the moment it finishes, so a crash never loses finished tests.
# (Per-example files stay separate: *_predictions.jsonl and *_generations.json.)
# ======================================================================================
"""
Evaluate ONE arm (a LoRA adapter folder, or the un-tuned base model) on:
  FLORES  EN->VI and VI->EN   (BLEU + chrF++)     <- new-language ability (translation)
  MLQA-vi                      (exact match, F1)   <- Vietnamese reading comprehension
  IFEval                       (strict / loose)    <- English instruction following (forgetting)

Colab setup (once per runtime, in normal notebook cells BEFORE running this script):
  !pip -q install peft sacrebleu langdetect immutabledict "nltk>=3.9.1"
  from google.colab import drive; drive.mount('/content/drive')
  from huggingface_hub import login; login()
First run with --quick (8 examples per test, ~2 min) to check the whole pipeline, then run without it.
"""
import argparse
import glob
import importlib
import json
import os
import re
import string
import subprocess
import sys
import time
import types
import unicodedata
from collections import Counter

import pandas as pd
import peft
import sacrebleu
import torch
import transformers
from langdetect import DetectorFactory, LangDetectException, detect
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

DetectorFactory.seed = 0  # langdetect is random by default; this makes language detection repeatable

# ============================================================ SETTINGS (defaults; the command line overrides them)
ADAPTER_ROOT = "/content/drive/MyDrive/adapters"
DATA_DIR = "/content/drive/MyDrive/SIFT_dataset"  # ifeval.jsonl, mlqa_vi.jsonl, flores.csv
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
SEED = 42
BS = {"flores": 32, "mlqa": 16, "ifeval": 50}  # generation batch sizes; lower them if you hit out-of-memory
MAX_NEW = {"flores": 200, "mlqa": 32, "ifeval": 1280}  # max generated tokens (IFEval needs room for long answers)
DTYPES = {"fp16": torch.float16, "bf16": torch.bfloat16}
DTYPE_NAMES = {"fp16": "float16", "bf16": "bfloat16"}
DEFAULT_CKPT = "ckpt_100pct"  # final checkpoint written by the training scripts
DEFAULT_OUT_NAME = "eval"

MLQA_PROMPT = (
    "Context: {context}\n\nQuestion: {question}\n\n"
    "Answer the question with a short phrase copied from the context. Do not explain."
)
FLORES_PROMPTS = {
    # "short" = the original eval wording. "train" = the wording the training scripts used.
    "short": {"en2vi": "Translate to Vietnamese: {}", "vi2en": "Translate to English: {}"},
    "train": {
        "en2vi": "Translate the following text to Vietnamese. Output only the translation.\n\n{}",
        "vi2en": "Translate the following text to English. Output only the translation.\n\n{}",
    },
}
QUICK_N = 8
SCHEMA = 2  # bump when the metrics change; saved results with another schema are rescored automatically
TEST_NAMES = ("flores_en2vi", "flores_vi2en", "mlqa_vi", "ifeval")

# filled in by setup_run() from the command line
ARM = None
N_FLORES = None  # None = use ALL rows in flores.csv; a number = fixed random sample of that size
QUICK = False  # True = 8 examples per test, saved to <out-name>_quick/ (pipeline check only)
FORCE = False  # True = redo tests even if their results already exist
DTYPE_KEY = "fp16"
FLORES_PROMPT = "short"
ARM_DIR = None  # <adapter-root>/<arm>
ADAPTER_DIR = None  # the folder that actually holds adapter_config.json (arm folder or a ckpt_* subfolder)
RESULT_DIR = None
RESULTS_PATH = None

model = tok = None  # filled by load_model()


# ============================================================ small helpers
def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    return rows[:QUICK_N] if QUICK else rows


def mean(xs):
    return sum(xs) / max(1, len(xs))


def pct(xs):
    """Percentage of True values in a list of booleans."""
    return round(100 * mean([bool(x) for x in xs]), 2)


def ci95(xs):
    """95% confidence half-width (in percentage points) for the mean of per-example scores in [0, 1].
    Read it as: score +/- this. Two arms whose difference is smaller than this are not clearly different."""
    n = len(xs)
    if n < 2:
        return 0.0
    m = mean(xs)
    var = sum((x - m) ** 2 for x in xs) / (n - 1)
    return round(100 * 1.96 * (var / n) ** 0.5, 2)


def lang_of(text):
    """Detected language code ('en', 'vi', ...) or None if the text is empty / undetectable."""
    try:
        return detect(text)
    except LangDetectException:
        return None


def gen_stats(outs, max_new):
    """Facts about the raw generations. truncated_pct is approximate (re-tokenised text): answers that hit the token cap."""
    lens = [len(ids) for ids in tok(outs, add_special_tokens=False).input_ids]
    return {
        "avg_gen_tokens": round(mean(lens), 1),
        "truncated_pct": pct([n >= max_new - 2 for n in lens]),
        "empty_pct": pct([not o.strip() for o in outs]),
    }


def pct_table(counts):
    """{key: [n, strict_hits, loose_hits]} -> {key: {"n", "strict", "loose"}} in percent."""
    return {k: {"n": n, "strict": round(100 * s / n, 2), "loose": round(100 * l / n, 2)}
            for k, (n, s, l) in sorted(counts.items())}


# ============================================================ the ONE results file
def load_results():
    if RESULTS_PATH and os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_results(results):
    """Atomic write (temp file, then rename) so an interrupted save can never corrupt results.json."""
    results["last_updated"] = time.strftime("%Y-%m-%d %H:%M")
    tmp = RESULTS_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    os.replace(tmp, RESULTS_PATH)


def run_info():
    """Everything needed to reproduce / compare this arm later, stored once at the top of results.json."""
    info = {
        "arm": ARM, "base_model": BASE_MODEL, "quick_mode": QUICK, "seed": SEED,
        "generation": {"decoding": "greedy", "dtype": DTYPE_NAMES[DTYPE_KEY], "max_new_tokens": dict(MAX_NEW),
                       "batch_size": dict(BS), "flores_prompt": FLORES_PROMPT},
        "flores_sample": "all rows" if N_FLORES is None else f"random {N_FLORES} (seed {SEED})",
        "environment": {
            "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
            "transformers": transformers.__version__, "peft": peft.__version__, "sacrebleu": sacrebleu.__version__,
        },
    }
    if ARM != "base":
        info["checkpoint"] = os.path.relpath(ADAPTER_DIR, ARM_DIR)  # "." = adapter files sit in the arm folder itself
        cfg = os.path.join(ADAPTER_DIR, "adapter_config.json")
        if os.path.exists(cfg):
            with open(cfg, encoding="utf-8") as f:
                c = json.load(f)
            info["lora"] = {k: c.get(k) for k in ("r", "lora_alpha", "lora_dropout", "target_modules", "use_rslora", "use_dora")}
    return info


# ============================================================ model + batched generation
def load_model():
    global model, tok
    assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU."
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    tok.padding_side = "left"  # required for batched generation
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPES[DTYPE_KEY]).to("cuda")
    if ARM != "base":
        assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")), (
            f"No adapter found in {ADAPTER_DIR} (expected adapter_config.json + adapter_model.safetensors). "
            f"Use --ckpt to pick a checkpoint folder inside {ARM_DIR}."
        )
        model = PeftModel.from_pretrained(model, ADAPTER_DIR)
        model = model.merge_and_unload()  # fold the adapter into the weights: same outputs, faster generation
        print(f"loaded base + adapter: {ADAPTER_DIR}")
    else:
        print("evaluating the BASE model (no adapter)")
    model.eval()


@torch.no_grad()
def generate(prompts, max_new_tokens, bs):
    """Greedy, batched, chat-templated generation. Returns one string per prompt, in the original order."""
    texts = [
        tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
        for p in prompts
    ]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)  # longest first: OOM shows up at once
    outs = [None] * len(texts)
    t0 = time.time()
    for s in range(0, len(order), bs):
        idx = order[s: s + bs]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
            pad_token_id=tok.pad_token_id,
        )
        for j, i in enumerate(idx):
            outs[i] = tok.decode(out[j, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
        print(f"   generated {min(s + bs, len(order))}/{len(order)}  ({time.time() - t0:.0f}s)")
    return outs


def generate_cached(name, prompts, max_new_tokens, bs):
    """Like generate(), but saves the raw outputs first, so a scoring bug can never cost you the generation time."""
    path = os.path.join(RESULT_DIR, f"{name}_generations.json")
    if os.path.exists(path) and not FORCE:
        with open(path, encoding="utf-8") as f:
            outs = json.load(f)
        if len(outs) == len(prompts):
            print(f"   reusing saved generations: {path}")
            return outs
    outs = generate(prompts, max_new_tokens, bs)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(outs, f, ensure_ascii=False)
    return outs


# ============================================================ MLQA scoring (Vietnamese, same normalisation as the official script)
PUNCT = {chr(i) for i in range(sys.maxunicode) if unicodedata.category(chr(i)).startswith("P")}.union(string.punctuation)


def norm_vi(s):
    s = "".join(ch for ch in s.lower() if ch not in PUNCT)
    s = re.sub(r"\b(của|là|cái|chiếc|những)\b", " ", s)
    return " ".join(s.split())


def f1_score(pred, gold):
    p, g = norm_vi(pred).split(), norm_vi(gold).split()
    same = sum((Counter(p) & Counter(g)).values())
    if same == 0:
        return 0.0
    prec, rec = same / len(p), same / len(g)
    return 2 * prec * rec / (prec + rec)


def em_score(pred, gold):
    return float(norm_vi(pred) == norm_vi(gold))


# ============================================================ IFEval checker (lm-evaluation-harness code, used standalone)
def setup_ifeval_checker():
    """Fetch lm-eval's IFEval checker files from PyPI WITHOUT installing lm-eval itself
    (that could change your Colab package versions), then import them."""
    root = "/content/lm_eval_ifeval"
    if not os.path.exists(f"{root}/lm_eval/tasks/ifeval/utils.py"):
        subprocess.run([sys.executable, "-m", "pip", "download", "lm_eval==0.4.13", "--no-deps",
                        "-d", "/content/lme_whl", "-q"], check=True)
        import zipfile
        whl = glob.glob("/content/lme_whl/lm_eval-*.whl")[0]
        with zipfile.ZipFile(whl) as z:
            for n in z.namelist():
                if n.startswith("lm_eval/tasks/ifeval/") and n.endswith(".py"):
                    z.extract(n, root)
    for name, sub in [("lm_eval", "lm_eval"), ("lm_eval.tasks", "lm_eval/tasks"),
                      ("lm_eval.tasks.ifeval", "lm_eval/tasks/ifeval")]:
        m = types.ModuleType(name)
        m.__path__ = [f"{root}/{sub}"]  # lets submodules import without running lm-eval's heavy __init__ files
        sys.modules[name] = m
    return importlib.import_module("lm_eval.tasks.ifeval.utils")


# ============================================================ the three tests
def load_flores():
    df = pd.read_csv(os.path.join(DATA_DIR, "flores.csv")).dropna(subset=["en", "vi"])
    if N_FLORES is not None and len(df) > N_FLORES:
        df = df.sample(n=N_FLORES, random_state=SEED)
    df = df.reset_index(drop=True)
    return df.head(QUICK_N) if QUICK else df


def run_flores(direction):
    df = load_flores()
    tmpl = FLORES_PROMPTS[FLORES_PROMPT][direction]
    if direction == "en2vi":
        src, ref, target = list(df.en), list(df.vi), "vi"
    else:
        src, ref, target = list(df.vi), list(df.en), "en"
    hyp = generate_cached(f"flores_{direction}", [tmpl.format(s) for s in src], MAX_NEW["flores"], BS["flores"])
    bleu = sacrebleu.corpus_bleu(hyp, [ref])
    chrf = sacrebleu.corpus_chrf(hyp, [ref], word_order=2)
    langs = [lang_of(h) for h in hyp]
    metrics = {
        "n": len(hyp),
        "bleu": round(bleu.score, 2),
        "chrfpp": round(chrf.score, 2),
        "bleu_length_ratio": round(bleu.ratio, 3),  # hypothesis length / reference length (1.0 = same length)
        "bleu_brevity_penalty": round(bleu.bp, 3),  # < 1.0 means outputs are too short and BLEU was reduced
        "target_language_pct": pct([lang == target for lang in langs]),  # output really is in the requested language
        "copied_source_pct": pct([h.strip() == s.strip() for h, s in zip(hyp, src)]),  # model just repeated the input
        "multiline_pct": pct(["\n" in h for h in hyp]),  # extra chatter such as "Here is the translation:"
    }
    metrics.update(gen_stats(hyp, MAX_NEW["flores"]))
    rows = [{"id": int(i), "source": s, "reference": r, "hypothesis": h, "lang": lang,
             "chrfpp": round(sacrebleu.sentence_chrf(h, [r], word_order=2).score, 2)}
            for i, s, r, h, lang in zip(df.id, src, ref, hyp, langs)]
    return metrics, rows


def run_mlqa():
    rows = load_jsonl(os.path.join(DATA_DIR, "mlqa_vi.jsonl"))
    prompts = [MLQA_PROMPT.format(context=r["context"], question=r["question"]) for r in rows]
    outs = generate_cached("mlqa_vi", prompts, MAX_NEW["mlqa"], BS["mlqa"])
    ems, f1s, in_ctx, pred_words, gold_words, out_rows = [], [], [], [], [], []
    for r, o in zip(rows, outs):
        pred = o.split("\n")[0].strip()
        golds = list(r["answers"]["text"])
        em, f1 = max(em_score(pred, g) for g in golds), max(f1_score(pred, g) for g in golds)
        p_norm = norm_vi(pred)
        found = bool(p_norm) and p_norm in norm_vi(r["context"])  # did it copy a span, as the prompt asks?
        ems.append(em)
        f1s.append(f1)
        in_ctx.append(found)
        pred_words.append(len(pred.split()))
        gold_words.append(len(golds[0].split()))
        out_rows.append({"id": r.get("id"), "question": r["question"], "gold": golds, "prediction": pred,
                         "em": em, "f1": f1, "in_context": found})
    metrics = {
        "n": len(rows),
        "exact_match": round(100 * mean(ems), 2), "exact_match_ci95": ci95(ems),
        "f1": round(100 * mean(f1s), 2), "f1_ci95": ci95(f1s),
        "answer_in_context_pct": pct(in_ctx),
        "avg_pred_words": round(mean(pred_words), 1), "avg_gold_words": round(mean(gold_words), 1),
        "multiline_pct": pct(["\n" in o.strip() for o in outs]),  # model explained instead of answering briefly
    }
    metrics.update(gen_stats(outs, MAX_NEW["mlqa"]))
    return metrics, out_rows


def fix_ints(kwargs):
    """The saved IFEval kwargs can contain 3.0 instead of 3 (pandas turns ints next to None into floats),
    which crashes the checker (e.g. list index 2.0). Turn whole-number floats back into ints."""
    return [{k: (int(v) if isinstance(v, float) and v.is_integer() else v) for k, v in d.items()} for d in kwargs]


def run_ifeval():
    utils = setup_ifeval_checker()
    rows = load_jsonl(os.path.join(DATA_DIR, "ifeval.jsonl"))
    outs = generate_cached("ifeval", [r["prompt"] for r in rows], MAX_NEW["ifeval"], BS["ifeval"])
    ps, pl, ins, inl, out_rows, errors = [], [], [], [], [], 0
    by_id, by_cat, langs = {}, {}, []
    for r, o in zip(rows, outs):
        doc = dict(r, kwargs=fix_ints(r["kwargs"]))
        try:
            res = utils.process_results(doc, [o])
        except Exception as e:  # one odd response must not kill the whole evaluation; it is counted and reported
            errors += 1
            print(f"   scoring error on key {r['key']}: {type(e).__name__}: {e}")
            n_inst = len(r["instruction_id_list"])
            res = {"prompt_level_strict_acc": False, "prompt_level_loose_acc": False,
                   "inst_level_strict_acc": [False] * n_inst, "inst_level_loose_acc": [False] * n_inst}
        ps.append(res["prompt_level_strict_acc"])
        pl.append(res["prompt_level_loose_acc"])
        ins += res["inst_level_strict_acc"]
        inl += res["inst_level_loose_acc"]
        # per instruction type, e.g. "length_constraints:number_words" and its category "length_constraints"
        for iid, s, l in zip(r["instruction_id_list"], res["inst_level_strict_acc"], res["inst_level_loose_acc"]):
            for table, key in ((by_id, iid), (by_cat, iid.split(":")[0])):
                c = table.setdefault(key, [0, 0, 0])
                c[0] += 1
                c[1] += int(s)
                c[2] += int(l)
        # English-drift check: skip prompts that explicitly ask for another language
        lang = None
        if "language:response_language" not in r["instruction_id_list"]:
            lang = lang_of(o)
            if lang:
                langs.append(lang)
        out_rows.append({"key": r["key"], "prompt": r["prompt"], "response": o, "lang": lang,
                         "strict": bool(res["prompt_level_strict_acc"]), "loose": bool(res["prompt_level_loose_acc"])})
    metrics = {
        "n": len(rows),
        "prompt_strict": round(100 * mean(ps), 2), "prompt_strict_ci95": ci95([float(x) for x in ps]),
        "prompt_loose": round(100 * mean(pl), 2), "prompt_loose_ci95": ci95([float(x) for x in pl]),
        "inst_strict": round(100 * mean(ins), 2), "inst_loose": round(100 * mean(inl), 2),
        "non_english_pct": pct([lg != "en" for lg in langs]),  # English prompts answered in another language
        "vietnamese_pct": pct([lg == "vi" for lg in langs]),  # ... specifically in Vietnamese (forgetting / drift)
        "scoring_errors": errors,
    }
    metrics.update(gen_stats(outs, MAX_NEW["ifeval"]))
    metrics["by_category"] = pct_table(by_cat)
    metrics["by_instruction"] = pct_table(by_id)
    if errors:
        print(f"   WARNING: {errors} responses could not be scored and were counted as failures")
    return metrics, out_rows


def cached(name, fn, expect_n=None):
    """Run one test and append its metrics to results.json. Skipped if a current-format result already exists."""
    saved = load_results().get(name)
    if saved is not None and not FORCE:
        if saved.get("schema") != SCHEMA:
            print(f"[{name}] saved result is from an older version of this script -> rescoring (saved generations are reused)")
        elif expect_n is not None and saved.get("n") != expect_n:
            print(f"[{name}] saved result has n={saved.get('n')} but n={expect_n} is needed -> rerunning")
        else:
            print(f"[{name}] already done -> loaded from {RESULTS_PATH}")
            return saved
    print(f"\n[{name}] running ...")
    t0 = time.time()
    metrics, rows = fn()
    metrics["seconds"] = round(time.time() - t0)
    metrics["finished_at"] = time.strftime("%Y-%m-%d %H:%M")
    metrics["schema"] = SCHEMA
    save_jsonl(rows, os.path.join(RESULT_DIR, f"{name}_predictions.jsonl"))
    results = load_results()  # re-read from disk, then add this test: results.json grows as tests complete
    results[name] = metrics
    save_results(results)
    short = {k: v for k, v in metrics.items() if not isinstance(v, dict)}  # keep the console line readable
    print(f"[{name}] {short}")
    print(f"   saved to {RESULTS_PATH}")
    print("   example output:", repr(str(rows[0].get("hypothesis") or rows[0].get("prediction") or rows[0].get("response"))[:150]))
    return metrics


# ============================================================ summary + command line
def print_summary():
    s = load_results()
    print("\n================ SUMMARY:", ARM, "(QUICK MODE)" if QUICK else "", "================")
    for k in TEST_NAMES:
        m = s.get(k)
        if m is None:
            continue
        if m.get("schema") != SCHEMA:
            print(f"{k:14s} (old format: rerun this test to refresh it)")
        elif k.startswith("flores"):
            print(f"{k:14s} BLEU {m['bleu']:6.2f}   chrF++ {m['chrfpp']:6.2f}   right language {m['target_language_pct']:5.1f}%   "
                  f"length ratio {m['bleu_length_ratio']:.2f}   (n={m['n']})")
        elif k == "mlqa_vi":
            print(f"{k:14s} EM   {m['exact_match']:6.2f} +/-{m['exact_match_ci95']:.1f}   F1 {m['f1']:6.2f} +/-{m['f1_ci95']:.1f}   "
                  f"answer in context {m['answer_in_context_pct']:5.1f}%   (n={m['n']})")
        else:
            print(f"{k:14s} prompt strict {m['prompt_strict']:.2f} +/-{m['prompt_strict_ci95']:.1f} / loose {m['prompt_loose']:.2f}   "
                  f"inst strict {m['inst_strict']:.2f} / loose {m['inst_loose']:.2f}   "
                  f"non-English {m['non_english_pct']:.1f}%   truncated {m['truncated_pct']:.1f}%   (n={m['n']})")
    print("saved to", RESULTS_PATH)


def parse_args(argv=None):
    p = argparse.ArgumentParser(description="Evaluate ONE arm (LoRA adapter folder or the base model).")
    p.add_argument("--arm", required=True,
                   help='adapter folder name inside --adapter-root, or "base" for the un-tuned model')
    p.add_argument("--tests", nargs="+", default=["all"], choices=["flores", "mlqa", "ifeval", "all"],
                   help="which evals to run (default: all)")
    p.add_argument("--quick", action="store_true", help="8 examples per test, saved to <out-name>_quick/")
    p.add_argument("--force", action="store_true", help="redo tests even if results already exist")
    p.add_argument("--n-flores", type=int, default=None,
                   help="FLORES sentences per direction (default: all rows in flores.csv)")
    p.add_argument("--adapter-root", default=ADAPTER_ROOT)
    p.add_argument("--data-dir", default=DATA_DIR)
    # ---- modular options (defaults keep the previous behaviour) ----
    p.add_argument("--ckpt", default=DEFAULT_CKPT,
                   help=f"checkpoint folder inside the arm (default {DEFAULT_CKPT}); if it does not exist, the arm "
                        "folder itself is used when the adapter files are there. Ignored for --arm base.")
    p.add_argument("--out-name", default=None,
                   help=f'results folder name inside the arm (default "{DEFAULT_OUT_NAME}"; "{DEFAULT_OUT_NAME}_<ckpt>" '
                        f"when you evaluate a checkpoint other than {DEFAULT_CKPT})")
    p.add_argument("--dtype", choices=["fp16", "bf16"], default="fp16",
                   help="generation precision (default fp16, which is the fast one on a T4)")
    p.add_argument("--base-model", default=BASE_MODEL)
    p.add_argument("--seed", type=int, default=SEED, help="seed for the FLORES random sample (with --n-flores)")
    p.add_argument("--bs-flores", type=int, default=BS["flores"])
    p.add_argument("--bs-mlqa", type=int, default=BS["mlqa"])
    p.add_argument("--bs-ifeval", type=int, default=BS["ifeval"])
    p.add_argument("--max-new-flores", type=int, default=MAX_NEW["flores"])
    p.add_argument("--max-new-mlqa", type=int, default=MAX_NEW["mlqa"])
    p.add_argument("--max-new-ifeval", type=int, default=MAX_NEW["ifeval"])
    p.add_argument("--flores-prompt", choices=list(FLORES_PROMPTS), default="short",
                   help='FLORES instruction wording: "short" (previous behaviour) or "train" (the wording the arms '
                        "were trained with). Use the SAME value for every arm you compare, including base.")
    return p.parse_args(argv)


def resolve_adapter_dir(arm_dir, ckpt):
    """The folder holding adapter_config.json: <arm>/<ckpt> if it is there, else the arm folder itself."""
    if ckpt and os.path.exists(os.path.join(arm_dir, ckpt, "adapter_config.json")):
        return os.path.join(arm_dir, ckpt)
    if os.path.exists(os.path.join(arm_dir, "adapter_config.json")):
        return arm_dir
    return os.path.join(arm_dir, ckpt) if ckpt else arm_dir  # not found: load_model() prints a helpful error


def setup_run(args):
    """Copy the command line into the module settings and work out the folders. Returns the list of tests to run."""
    global ARM, ADAPTER_ROOT, DATA_DIR, BASE_MODEL, SEED, N_FLORES, QUICK, FORCE
    global DTYPE_KEY, FLORES_PROMPT, ARM_DIR, ADAPTER_DIR, RESULT_DIR, RESULTS_PATH
    ARM, ADAPTER_ROOT, DATA_DIR, BASE_MODEL, SEED = args.arm, args.adapter_root, args.data_dir, args.base_model, args.seed
    N_FLORES, QUICK, FORCE = args.n_flores, args.quick, args.force
    DTYPE_KEY, FLORES_PROMPT = args.dtype, args.flores_prompt
    BS.update(flores=args.bs_flores, mlqa=args.bs_mlqa, ifeval=args.bs_ifeval)
    MAX_NEW.update(flores=args.max_new_flores, mlqa=args.max_new_mlqa, ifeval=args.max_new_ifeval)

    ARM_DIR = os.path.join(ADAPTER_ROOT, ARM)
    ADAPTER_DIR = ARM_DIR if ARM == "base" else resolve_adapter_dir(ARM_DIR, args.ckpt)
    folder = os.path.basename(ADAPTER_DIR)
    default_ckpt = ARM == "base" or ADAPTER_DIR == ARM_DIR or folder == DEFAULT_CKPT
    out_name = args.out_name or (DEFAULT_OUT_NAME if default_ckpt else f"{DEFAULT_OUT_NAME}_{folder}")
    RESULT_DIR = os.path.join(ARM_DIR, out_name + ("_quick" if QUICK else ""))
    RESULTS_PATH = os.path.join(RESULT_DIR, "results.json")
    return ["flores", "mlqa", "ifeval"] if "all" in args.tests else args.tests


def main():
    tests = setup_run(parse_args())

    os.makedirs(RESULT_DIR, exist_ok=True)
    load_model()

    # Refuse to mix results made with different generation settings in the same folder.
    info = run_info()
    prev = load_results().get("generation")
    if prev and not FORCE:
        changed = [k for k in ("decoding", "dtype", "max_new_tokens", "flores_prompt")
                   if k in prev and prev[k] != info["generation"][k]]
        if changed:
            raise SystemExit(
                f"{RESULTS_PATH} was made with different settings ({', '.join(changed)}). "
                "Use --force to redo the tests, or --out-name to write to a new folder."
            )

    results = load_results()
    results.pop("date", None)  # key from the previous version of this script
    results.update(info)
    save_results(results)

    if "flores" in tests:
        n_flores = len(load_flores())
        cached("flores_en2vi", lambda: run_flores("en2vi"), expect_n=n_flores)
        cached("flores_vi2en", lambda: run_flores("vi2en"), expect_n=n_flores)
    if "mlqa" in tests:
        cached("mlqa_vi", run_mlqa)
    if "ifeval" in tests:
        cached("ifeval", run_ifeval)

    print_summary()


if __name__ == "__main__":
    main()


Overwriting eval.py


In [ ]:
!python eval.py --arm plain_fine_tuning

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 146/146 [00:02<00:00, 72.54it/s]
loaded base + adapter: /content/drive/MyDrive/adapters/plain_fine_tuning/ckpt_100pct

[flores_en2vi] running ...
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
   generated 32/500  (3s)
   generated 64/500  (10s)
   generated 96/500  (11s)
   generated 128/500  (13s)
   generated 160/500  (15s)
   generated 192/500  (21s)
   generated 224/500  (23s)
   generated 256/500  (24s)
   generated 288/500  (25s)
   generated 320/500  (27s)
   generated 352/500  (28s)
   generated 384/500  (29s)
   generated 416